# LangGraph Pipeline Test — Direct Invocation (No Server)

Tests the core generation pipeline by invoking the LangGraph graph directly.
No HTTP server, no Docker — pure pipeline verification.

**Test Plan:**
1. Import & config verification
2. Smoke test — single topic, small batch (5 quiz)
3. Medium batch — 15 quiz (lesson scale)
4. Large batch — 30 quiz (Kahoot scale)
5. Multi game-type — quiz + flashcard + fill_blank
6. Multi-chapter topic — broad coverage
7. Difficulty levels — easy / medium / hard
8. STEM subject coverage — Math, Physics, Chemistry
9. Quality metrics summary

**Industry Benchmarks:**
- Kahoot: 20-30 candidate questions per topic
- Khan Academy: generate 2-3x needed, curate down
- Duolingo: ~200 candidates per lesson, show ~14
- ASEE best practice: generate 2-3x required, prune

## 1. Setup & Configuration

In [1]:
import sys, json, time
from pathlib import Path

project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import get_settings
from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope,
    GameContentResponse, QuizQuestion, Flashcard, FillBlankQuestion,
)
from src.graph.builder import compile_graph
from src.graph.state import MAX_REVIEW_ITERATIONS

settings = get_settings()
print(f'Project: {settings.gcp_project_id}')
print(f'Generation model: {settings.generation_model} @ {settings.generation_model_location or settings.gcp_location}')
print(f'Review model: {settings.review_model} @ {settings.review_model_location}')
print(f'Max review iterations: {MAX_REVIEW_ITERATIONS}')

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: green-mercury-485016-n1
Generation model: gemini-2.5-flash @ asia-southeast1
Review model: gemini-3.1-flash-lite-preview @ global
Max review iterations: 3


In [2]:
# ── Helpers ──

def pp(data):
    if hasattr(data, 'model_dump'):
        data = data.model_dump()
    print(json.dumps(data, ensure_ascii=False, indent=2, default=str))

def make_request(
    topic: str,
    num_questions: int = 10,
    game_types: list[str] | None = None,
    difficulty: str = 'comprehension',
    doc_scope: str = 'system',
) -> GenerationRequest:
    gts = [GameType(g) for g in (game_types or ['quiz'])]
    return GenerationRequest(
        user_id='pipeline_test',
        topic=topic,
        game_types=gts,
        num_questions=num_questions,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope(doc_scope),
    )

async def run_pipeline(request: GenerationRequest) -> dict:
    """Run pipeline with fresh compile (no singleton cache)."""
    app = compile_graph()
    initial_state = {
        'request': request,
        'doc_scope': request.doc_scope.value,
        'iteration_count': 0,
        'rejected_items': [],
        'errors': [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0
    return {**result, '_elapsed': elapsed}

def summarize(result: dict, label: str = ''):
    prefix = f'[{label}] ' if label else ''
    output: GameContentResponse | None = result.get('final_output')
    errors = result.get('errors', [])
    elapsed = result.get('_elapsed', 0)
    iterations = result.get('iteration_count', 0)

    if not output:
        print(f'{prefix}❌ No output. Errors: {errors}')
        return None

    c = output.content
    m = output.metadata
    qn, fn, bn = len(c.quiz), len(c.flashcard), len(c.fill_blank)
    total = qn + fn + bn

    print(f'{prefix}✅ {total} items in {elapsed:.1f}s ({iterations} iter)')
    print(f'  quiz={qn}  flashcard={fn}  fill_blank={bn}')
    print(f'  generated={m.total_generated}  passed={m.total_passed_review}  rejected={m.total_rejected}')
    if errors:
        print(f'  ⚠️ errors: {errors}')
    return {'quiz': qn, 'fc': fn, 'fb': bn, 'total': total, 'elapsed': elapsed,
            'generated': m.total_generated, 'passed': m.total_passed_review, 'rejected': m.total_rejected}

ALL = {}  # accumulate results
print(f'DifficultyLevel values: {[d.value for d in DifficultyLevel]}')
print('Helpers ready ✓')

DifficultyLevel values: ['recall', 'comprehension', 'application', 'high_application']
Helpers ready ✓


## 2. Smoke Test — 5 Quiz Questions

In [3]:
req = make_request('Đạo hàm', num_questions=5, game_types=['quiz'])
res = await run_pipeline(req)
ALL['smoke_5q'] = summarize(res, 'Smoke 5Q')

out = res['final_output']
assert out is not None, 'No output'
assert len(out.content.quiz) >= 3, f'Expected >=3 quiz, got {len(out.content.quiz)}'

q = out.content.quiz[0]
assert len(q.options) == 4
assert any(o.is_correct for o in q.options)
assert q.question and q.explanation

print(f'\nSample: {q.question[:120]}')
print(f'Correct: option {q.correct_answer_index}')
print(f'Trace: {(q.computation_trace or "none")[:120]}')
print('\n✓ Smoke test PASSED')

2026-03-27 09:16:59 [info     ] creating_graph                
2026-03-27 09:16:59 [info     ] graph_created                 
2026-03-27 09:16:59 [info     ] graph_compiled_without_checkpointer
2026-03-27 09:16:59 [info     ] supervisor_analyzing           doc_scope=system topic='Đạo hàm' user_id=pipeline_test
2026-03-27 09:16:59 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-27 09:16:59 [info     ] rate_limiter_init_bucket       capacity=5.0 refill_rate=1.0 rpm=60
2026-03-27 09:16:59 [info     ] rate_limiter_init_semaphore    max_concurrent=10


/home/sakana/Code/ManifoldsTeam/AIServices/src/services/llm.py:99: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  return ChatVertexAI(


2026-03-27 09:17:01 [info     ] supervisor_classified          content_type=math
2026-03-27 09:17:01 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-27 09:17:01 [info     ] math_agent_starting            doc_scope=system num_questions=5 num_to_generate=7 topic='Đạo hàm' user_id=pipeline_test
2026-03-27 09:17:01 [info     ] retrieving_context             doc_scope=system query='Đạo hàm comprehension level educational content' user_id=pipeline_test
2026-03-27 09:17:01 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=pipeline_test
2026-03-27 09:17:01 [debug    ] retrieving_documents           doc_scope=system query='Đạo hàm comprehension level educational content'


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:17:03 [debug    ] retrieved_documents            count=30
2026-03-27 09:17:03 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:17:03 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:17:03 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:18:03 [info     ] math_agent_code_execution_done code_traces_count=10 generation_attempt=1 text_length=9148
2026-03-27 09:18:03 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:18:25 [warning  ] math_agent_structured_output_returned_none attempt=1 batch=single
2026-03-27 09:18:44 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=7
2026-03-27 09:18:44 [info     ] math_agent_generated           items_count=7
2026-03-27 09:18:44 [info     ] reviewer_reviewing             items_count

## 3. Medium Batch — 15 Quiz (Lesson Scale)

In [4]:
req = make_request('Phương trình bậc hai', num_questions=15, game_types=['quiz'])
res = await run_pipeline(req)
ALL['medium_15q'] = summarize(res, '15Q Quiz')

out = res['final_output']
assert out is not None
assert len(out.content.quiz) >= 10, f'Expected >=10, got {len(out.content.quiz)}'

questions = [q.question for q in out.content.quiz]
assert len(set(questions)) == len(questions), 'Duplicate questions!'
print(f'Unique questions: {len(set(questions))}/{len(questions)}')

print('\n✓ Medium batch PASSED')

2026-03-27 09:19:11 [info     ] creating_graph                
2026-03-27 09:19:11 [info     ] graph_created                 
2026-03-27 09:19:11 [info     ] graph_compiled_without_checkpointer
2026-03-27 09:19:11 [info     ] supervisor_analyzing           doc_scope=system topic='Phương trình bậc hai' user_id=pipeline_test
2026-03-27 09:19:11 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-27 09:19:13 [info     ] supervisor_classified          content_type=math
2026-03-27 09:19:13 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-27 09:19:13 [info     ] math_agent_starting            doc_scope=system num_questions=15 num_to_generate=20 topic='Phương trình bậc hai' user_id=pipeline_test
2026-03-27 09:19:13 [info     ] retrieving_context             doc_scope=system query='Phương trình bậc hai comprehension level educational content' user_id=pipeline_test
2026-03-

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:19:15 [debug    ] retrieved_documents            count=24
2026-03-27 09:19:15 [info     ] context_retrieved              chunks=24 sources=24
2026-03-27 09:19:15 [info     ] math_agent_context_retrieved   context_chunks=24
2026-03-27 09:19:15 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:20:28 [info     ] math_agent_code_execution_done code_traces_count=22 generation_attempt=1 text_length=14256
2026-03-27 09:20:28 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:20:40 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=6
2026-03-27 09:20:40 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-27 09:20:41 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-27 09:20:41 [info     ] math_agent_parse_succeeded     generation

## 4. Large Batch — 30 Quiz (Kahoot Scale)

Kahoot generates 20-30 candidate questions per topic.

In [5]:
req = make_request('Tích phân', num_questions=30, game_types=['quiz'])
res = await run_pipeline(req)
ALL['large_30q'] = summarize(res, '30Q Quiz')

out = res['final_output']
assert out is not None
n = len(out.content.quiz)
assert n >= 21, f'Expected >=21 (70% of 30), got {n}'

questions = [q.question for q in out.content.quiz]
print(f'Unique: {len(set(questions))}/{len(questions)}')

for idx in [0, n//2, -1]:
    q = out.content.quiz[idx]
    print(f'\n[{idx}] {q.question[:100]}')
    print(f'    topic: {q.topic}')

print('\n✓ Large batch PASSED')

2026-03-27 09:21:21 [info     ] creating_graph                
2026-03-27 09:21:21 [info     ] graph_created                 
2026-03-27 09:21:21 [info     ] graph_compiled_without_checkpointer
2026-03-27 09:21:21 [info     ] supervisor_analyzing           doc_scope=system topic='Tích phân' user_id=pipeline_test
2026-03-27 09:21:21 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-27 09:21:22 [info     ] supervisor_classified          content_type=math
2026-03-27 09:21:22 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-27 09:21:22 [info     ] math_agent_starting            doc_scope=system num_questions=30 num_to_generate=39 topic='Tích phân' user_id=pipeline_test
2026-03-27 09:21:22 [info     ] retrieving_context             doc_scope=system query='Tích phân comprehension level educational content' user_id=pipeline_test
2026-03-27 09:21:22 [debug    ] creating_

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:21:27 [debug    ] retrieved_documents            count=29
2026-03-27 09:21:27 [info     ] context_retrieved              chunks=29 sources=29
2026-03-27 09:21:27 [info     ] math_agent_context_retrieved   context_chunks=29
2026-03-27 09:21:27 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:23:11 [info     ] math_agent_code_execution_done code_traces_count=26 generation_attempt=1 text_length=16928
2026-03-27 09:23:11 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:23:18 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-27 09:23:19 [info     ] math_agent_batch_parsed        attempt=1 batch=6 items=4
2026-03-27 09:23:20 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-27 09:23:21 [info     ] math_agent_batch_parsed        attempt=1 

## 5. Multi Game-Type — quiz + flashcard + fill_blank

Each reviewed item is formatted into ALL requested game types.
15 items × 3 types → ~45 total formatted items.

In [6]:
req = make_request(
    'Định luật Newton',
    num_questions=15,
    game_types=['quiz', 'flashcard', 'fill_blank'],
)
res = await run_pipeline(req)
ALL['multi_type'] = summarize(res, 'Multi-type 15Q')

out = res['final_output']
assert out is not None
qn = len(out.content.quiz)
fn = len(out.content.flashcard)
bn = len(out.content.fill_blank)
assert qn >= 10, f'Quiz: {qn}'
assert fn >= 10, f'Flashcard: {fn}'
assert bn >= 10, f'Fill-blank: {bn}'

# Validate flashcard
fc = out.content.flashcard[0]
assert fc.front and fc.back
print(f'\nFlashcard: {fc.front[:80]} → {fc.back[:80]}')

# Validate fill-blank
fb = out.content.fill_blank[0]
assert fb.template and fb.blanks
print(f'Fill-blank: {fb.template[:80]}')
print(f'  Blanks: {[(b.position, b.correct_answer) for b in fb.blanks]}')

print('\n✓ Multi game-type PASSED')

2026-03-27 09:24:23 [info     ] creating_graph                
2026-03-27 09:24:23 [info     ] graph_created                 
2026-03-27 09:24:23 [info     ] graph_compiled_without_checkpointer
2026-03-27 09:24:23 [info     ] supervisor_analyzing           doc_scope=system topic='Định luật Newton' user_id=pipeline_test
2026-03-27 09:24:23 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-27 09:24:26 [info     ] supervisor_classified          content_type=math
2026-03-27 09:24:26 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-27 09:24:26 [info     ] math_agent_starting            doc_scope=system num_questions=15 num_to_generate=20 topic='Định luật Newton' user_id=pipeline_test
2026-03-27 09:24:26 [info     ] retrieving_context             doc_scope=system query='Định luật Newton comprehension level educational content' user_id=pipeline_test
2026-03-27 09:24:26 

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:24:30 [debug    ] retrieved_documents            count=30
2026-03-27 09:24:30 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:24:30 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:24:30 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:24:53 [info     ] math_agent_code_execution_done code_traces_count=2 generation_attempt=1 text_length=14094
2026-03-27 09:24:53 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:25:02 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-27 09:25:02 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=6
2026-03-27 09:25:03 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-27 09:25:03 [info     ] math_agent_parse_succeeded     generation_

## 6. Multi-Chapter Topic — Broad Coverage

A broad topic like "Hàm số" spans chapters:
- Hàm số bậc hai (lớp 10)
- Hàm số mũ & logarit (lớp 12)
- Đạo hàm & ứng dụng (lớp 11-12)

The pipeline should cover sub-topics within the broader topic.

In [7]:
BROAD = [
    ('Hàm số và đồ thị', 20),
    ('Vectơ và hình học không gian', 15),
]

for topic, num_q in BROAD:
    print(f'\n{"="*50}')
    print(f'{topic} ({num_q}Q)')
    print('='*50)

    req = make_request(topic, num_questions=num_q, game_types=['quiz', 'flashcard'])
    res = await run_pipeline(req)
    key = f'broad_{topic[:8]}'
    ALL[key] = summarize(res, topic)

    out = res['final_output']
    if out and out.content.quiz:
        topics = [q.topic for q in out.content.quiz]
        dist = {}
        for t in topics:
            dist[t] = dist.get(t, 0) + 1
        print('Sub-topic distribution:')
        for t, c in sorted(dist.items(), key=lambda x: -x[1]):
            print(f'  {t}: {c}')

print('\n✓ Multi-chapter test completed')


Hàm số và đồ thị (20Q)
2026-03-27 09:25:49 [info     ] creating_graph                
2026-03-27 09:25:49 [info     ] graph_created                 
2026-03-27 09:25:49 [info     ] graph_compiled_without_checkpointer
2026-03-27 09:25:49 [info     ] supervisor_analyzing           doc_scope=system topic='Hàm số và đồ thị' user_id=pipeline_test
2026-03-27 09:25:49 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-27 09:25:51 [info     ] supervisor_classified          content_type=math
2026-03-27 09:25:51 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-27 09:25:51 [info     ] math_agent_starting            doc_scope=system num_questions=20 num_to_generate=26 topic='Hàm số và đồ thị' user_id=pipeline_test
2026-03-27 09:25:51 [info     ] retrieving_context             doc_scope=system query='Hàm số và đồ thị comprehension level educational content' user_id=pipeline_t

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:25:54 [debug    ] retrieved_documents            count=30
2026-03-27 09:25:54 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:25:54 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:25:54 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:28:26 [info     ] math_agent_code_execution_done code_traces_count=24 generation_attempt=1 text_length=17852
2026-03-27 09:28:26 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:28:37 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=7
2026-03-27 09:28:37 [info     ] math_agent_batch_parsed        attempt=1 batch=4 items=5
2026-03-27 09:28:40 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-27 09:28:40 [info     ] math_agent_batch_parsed        attempt=1 

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:29:27 [debug    ] retrieved_documents            count=30
2026-03-27 09:29:27 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:29:27 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:29:27 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:29:53 [info     ] math_agent_code_execution_done code_traces_count=2 generation_attempt=1 text_length=12169
2026-03-27 09:29:53 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:30:02 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-27 09:30:02 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-27 09:30:07 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=6
2026-03-27 09:30:07 [info     ] math_agent_parse_succeeded     generation_

## 7. Difficulty Levels — recall / comprehension / application / high_application

In [8]:
TOPIC = 'Phản ứng oxi hóa khử'

for diff in ['recall', 'comprehension', 'application', 'high_application']:
    print(f'\n--- {diff} ---')
    req = make_request(TOPIC, num_questions=10, difficulty=diff, game_types=['quiz'])
    res = await run_pipeline(req)
    ALL[f'diff_{diff}'] = summarize(res, diff)

    out = res['final_output']
    if out and out.content.quiz:
        q = out.content.quiz[0]
        print(f'  Sample: {q.question[:100]}')
        print(f'  Difficulty: {q.difficulty}')

print('\n✓ Difficulty test completed')


--- recall ---
2026-03-27 09:30:49 [info     ] creating_graph                
2026-03-27 09:30:49 [info     ] graph_created                 
2026-03-27 09:30:49 [info     ] graph_compiled_without_checkpointer
2026-03-27 09:30:49 [info     ] supervisor_analyzing           doc_scope=system topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-27 09:30:49 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-27 09:30:51 [info     ] supervisor_classified          content_type=math
2026-03-27 09:30:51 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-27 09:30:51 [info     ] math_agent_starting            doc_scope=system num_questions=10 num_to_generate=13 topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-27 09:30:51 [info     ] retrieving_context             doc_scope=system query='Phản ứng oxi hóa khử recall level educational content' user_id=pipeline_test

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:30:53 [debug    ] retrieved_documents            count=30
2026-03-27 09:30:53 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:30:53 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:30:53 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:31:11 [info     ] math_agent_code_execution_done code_traces_count=2 generation_attempt=1 text_length=4873
2026-03-27 09:31:11 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:31:16 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-27 09:31:23 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=6
2026-03-27 09:31:23 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=13
2026-03-27 09:31:23 [info     ] math_agent_generated           items_co

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:31:39 [debug    ] retrieved_documents            count=30
2026-03-27 09:31:39 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:31:39 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:31:39 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:33:03 [info     ] math_agent_code_execution_done code_traces_count=6 generation_attempt=1 text_length=8930
2026-03-27 09:33:03 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:33:13 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-27 09:33:18 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=6
2026-03-27 09:33:18 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=13
2026-03-27 09:33:18 [info     ] math_agent_generated           items_co

...

2026-03-27 09:37:50 [debug    ] retrieved_documents            count=30
2026-03-27 09:37:50 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:37:50 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:37:50 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:38:21 [info     ] math_agent_code_execution_done code_traces_count=0 generation_attempt=1 text_length=845
2026-03-27 09:38:21 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:38:33 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=0
2026-03-27 09:38:52 [warning  ] math_agent_batch_parse_retry   attempt=1 batch=1
2026-03-27 09:39:10 [info     ] math_agent_batch_parsed        attempt=2 batch=1 items=0
2026-03-27 09:39:10 [warning  ] math_agent_parse_failed_retrying generation_attempt=

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:40:11 [debug    ] retrieved_documents            count=30
2026-03-27 09:40:11 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:40:11 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:40:11 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:41:30 [info     ] math_agent_code_execution_done code_traces_count=6 generation_attempt=1 text_length=28134
2026-03-27 09:41:30 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:41:48 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=6
2026-03-27 09:41:57 [warning  ] math_agent_batch_parse_retry   attempt=1 batch=1
2026-03-27 09:42:21 [warning  ] math_agent_batch_parse_retry   attempt=2 batch=1
2026-03-27 09:42:21 [warning  ] math_agent_batch_parse_failed  batch=1
2026-03-27 09:42:21

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:42:30 [debug    ] retrieved_documents            count=30
2026-03-27 09:42:30 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:42:30 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:42:30 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:43:01 [info     ] math_agent_code_execution_done code_traces_count=0 generation_attempt=1 text_length=2871
2026-03-27 09:43:01 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:43:12 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=6
2026-03-27 09:43:20 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-27 09:43:20 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=13
2026-03-27 09:43:20 [info     ] math_agent_generated           items_co

## 8. STEM Subject Coverage (20 questions each, all game types)

Real-world usage: generate content for an entire lesson across subjects.

In [9]:
SUBJECTS = [
    ('Toán', 'Giải hệ phương trình bậc nhất hai ẩn', 20),
    ('Lý', 'Chuyển động thẳng đều và biến đổi đều', 20),
    ('Hóa', 'Cấu hình electron và bảng tuần hoàn', 20),
]

for subj, topic, nq in SUBJECTS:
    print(f'\n{"="*50}')
    print(f'[{subj}] {topic} ({nq}Q × 3 types)')
    print('='*50)

    req = make_request(topic, num_questions=nq, game_types=['quiz', 'flashcard', 'fill_blank'])
    res = await run_pipeline(req)
    ALL[f'subj_{subj}'] = summarize(res, subj)

    out = res['final_output']
    if out:
        traces = sum(1 for q in out.content.quiz if q.computation_trace)
        print(f'  Computation traces: {traces}/{len(out.content.quiz)}')

print('\n✓ Subject coverage test completed')


[Toán] Giải hệ phương trình bậc nhất hai ẩn (20Q × 3 types)
2026-03-27 09:43:44 [info     ] creating_graph                
2026-03-27 09:43:44 [info     ] graph_created                 
2026-03-27 09:43:44 [info     ] graph_compiled_without_checkpointer
2026-03-27 09:43:44 [info     ] supervisor_analyzing           doc_scope=system topic='Giải hệ phương trình bậc nhất hai ẩn' user_id=pipeline_test
2026-03-27 09:43:44 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-27 09:43:46 [info     ] supervisor_classified          content_type=math
2026-03-27 09:43:46 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-27 09:43:46 [info     ] math_agent_starting            doc_scope=system num_questions=20 num_to_generate=26 topic='Giải hệ phương trình bậc nhất hai ẩn' user_id=pipeline_test
2026-03-27 09:43:46 [info     ] retrieving_context             doc_scope=system query=

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:43:50 [debug    ] retrieved_documents            count=30
2026-03-27 09:43:50 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:43:50 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:43:50 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:45:10 [info     ] math_agent_code_execution_done code_traces_count=8 generation_attempt=1 text_length=17296
2026-03-27 09:45:10 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:45:18 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-27 09:45:20 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-27 09:45:26 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=7
2026-03-27 09:45:32 [warning  ] math_agent_batch_parse_retry   attempt=1 b

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:46:59 [debug    ] retrieved_documents            count=30
2026-03-27 09:46:59 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:46:59 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:46:59 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:47:46 [info     ] math_agent_code_execution_done code_traces_count=22 generation_attempt=1 text_length=13128
2026-03-27 09:47:46 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:47:54 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-27 09:47:55 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-27 09:47:55 [info     ] math_agent_batch_parsed        attempt=1 batch=4 items=5
2026-03-27 09:47:56 [info     ] math_agent_batch_parsed        attempt=1 

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-27 09:48:33 [debug    ] retrieved_documents            count=30
2026-03-27 09:48:33 [info     ] context_retrieved              chunks=30 sources=30
2026-03-27 09:48:33 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-27 09:48:33 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-27 09:50:10 [info     ] math_agent_code_execution_done code_traces_count=6 generation_attempt=1 text_length=15176
2026-03-27 09:50:10 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-27 09:50:18 [info     ] math_agent_batch_parsed        attempt=1 batch=4 items=5
2026-03-27 09:50:32 [warning  ] math_agent_batch_parse_retry   attempt=1 batch=1
2026-03-27 09:50:33 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-27 09:50:34 [warning  ] math_agent_batch_parse_retry   attempt=1 batch=3
2

## 9. Quality Metrics Summary

In [10]:
print('=' * 85)
print('PIPELINE TEST SUMMARY')
print('=' * 85)
print(f'{"Test":<22} {"Total":>6} {"Quiz":>5} {"FC":>4} {"FB":>4} {"Gen":>5} {"Pass%":>6} {"Time":>7}')
print('-' * 85)

total_items = 0
total_time = 0.0
pass_rates = []

for name, s in ALL.items():
    if s is None:
        print(f'{name:<22} FAILED')
        continue
    pr = s['passed'] / s['generated'] * 100 if s['generated'] > 0 else 0
    pass_rates.append(pr)
    total_items += s['total']
    total_time += s['elapsed']
    print(f'{name:<22} {s["total"]:>6} {s["quiz"]:>5} {s["fc"]:>4} {s["fb"]:>4} {s["generated"]:>5} {pr:>5.1f}% {s["elapsed"]:>6.1f}s')

print('-' * 85)
avg_pr = sum(pass_rates) / len(pass_rates) if pass_rates else 0
print(f'{"TOTAL":<22} {total_items:>6} {"":>5} {"":>4} {"":>4} {"":>5} {avg_pr:>5.1f}% {total_time:>6.1f}s')

print(f'\nTests: {len(ALL)} | Items: {total_items} | Avg pass rate: {avg_pr:.1f}% | Time: {total_time:.0f}s ({total_time/60:.1f}min)')

failed = [k for k, v in ALL.items() if v is None]
if failed:
    print(f'❌ Failed: {failed}')
else:
    print('✅ All pipeline tests passed')

PIPELINE TEST SUMMARY
Test                    Total  Quiz   FC   FB   Gen  Pass%    Time
-------------------------------------------------------------------------------------
smoke_5q                    5     5    0    0     5 100.0%  123.3s
medium_15q                 15    15    0    0    15 100.0%  119.7s
large_30q                  30    30    0    0    30 100.0%  170.4s
multi_type                 45    15   15   15    15 100.0%   75.3s
broad_Hàm số v             40    20   20    0    20 100.0%  212.7s
broad_Vectơ và             30    15   15    0    15 100.0%   76.5s
diff_recall                10    10    0    0    10 100.0%   45.6s
diff_comprehension         10    10    0    0    11  90.9%  125.9s
diff_application           10    10    0    0    10 100.0%  243.9s
diff_high_application       0     0    0    0     0   0.0%  339.1s
subj_Toán                  60    20   20   20    20 100.0%  186.1s
subj_Lý                    60    20   20   20    20 100.0%   98.5s
subj_Hóa             

In [16]:
# Check diff_high_application in detail
d = ALL['diff_high_application']
print(f"high_application: total={d['total']}, gen={d['generated']}, passed={d['passed']}, rejected={d['rejected']}, time={d['elapsed']:.1f}s")
print(f"quiz={d['quiz']}, fc={d['fc']}, fb={d['fb']}")
print()
# Also check multi_type rate anomaly
for name in ['multi_type', 'broad_Hàm số v', 'broad_Vectơ và', 'subj_Toán', 'subj_Lý', 'subj_Hóa']:
    d = ALL[name]
    print(f"{name}: total={d['total']}, gen={d['generated']}, passed={d['passed']}, rejected={d['rejected']}")

high_application: total=0, gen=0, passed=0, rejected=32, time=339.1s
quiz=0, fc=0, fb=0

multi_type: total=45, gen=15, passed=15, rejected=0
broad_Hàm số v: total=40, gen=20, passed=20, rejected=0
broad_Vectơ và: total=30, gen=15, passed=15, rejected=0
subj_Toán: total=60, gen=20, passed=20, rejected=0
subj_Lý: total=60, gen=20, passed=20, rejected=0
subj_Hóa: total=60, gen=20, passed=20, rejected=0
